In [1]:
import pandas as pd


In [2]:
import numpy as np


In [3]:
from scipy.sparse import csr_matrix

In [4]:
from implicit.evaluation import train_test_split

In [5]:
import implicit

In [6]:
from implicit.evaluation import precision_at_k

In [7]:
train = pd.read_csv("train.csv")

In [8]:
train.head()

,msno,song_id,source_system_tab,source_screen_name,source_type,target
0,FGtllVqz18RPiwJj/edr2gV78zirAiY/9SmYvia+kCg=,BBzumQNXUHKdEBOB7mAJuzok+IJA1c2Ryg/yzTF6tik=,explore,Explore,online-playlist,1
1,Xumu+NIjS6QYVxDS4/t3SawvJ7viT9hPKXmf0RtLNx8=,bhp/MpSNoqoxOIB+/l8WPqu6jldth4DIpCm3ayXnJqM=,my library,Local playlist more,local-playlist,1
2,Xumu+NIjS6QYVxDS4/t3SawvJ7viT9hPKXmf0RtLNx8=,JNWfrrC7zNN7BdMpsISKa4Mw+xVJYNnxXh3/Epw7QgY=,my library,Local playlist more,local-playlist,1
3,Xumu+NIjS6QYVxDS4/t3SawvJ7viT9hPKXmf0RtLNx8=,2A87tzfnJTSWqD7gIZHisolhe4DMdzkbd6LzO1KHjNs=,my library,Local playlist more,local-playlist,1
4,FGtllVqz18RPiwJj/edr2gV78zirAiY/9SmYvia+kCg=,3qm6XTZ6MOCU11x8FIVbAGH5l5uMkT3/ZalWG1oo2Gc=,explore,Explore,online-playlist,1


In [9]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7377418 entries, 0 to 7377417
Data columns (total 6 columns):
 #   Column              Dtype 
---  ------              ----- 
 0   msno                object
 1   song_id             object
 2   source_system_tab   object
 3   source_screen_name  object
 4   source_type         object
 5   target              int64 
dtypes: int64(1), object(5)
memory usage: 337.7+ MB


In [10]:
train.describe()

,target
count,7.377418e+06
mean,5.035171e-01
std,4.999877e-01
min,0.000000e+00
25%,0.000000e+00
50%,1.000000e+00
75%,1.000000e+00
max,1.000000e+00


In [11]:
train['msno'].nunique()

30755

In [12]:
train['song_id'].nunique()

359966

In [13]:
df = train[["msno", "song_id", "target"]].copy()

In [14]:
df.isnull().sum()

msno       0
song_id    0
target     0
dtype: int64

In [15]:
df = df[df["target"] == 1]

In [16]:
df = df.drop_duplicates(["msno", "song_id"])

In [17]:
df.head()

,msno,song_id,target
0,FGtllVqz18RPiwJj/edr2gV78zirAiY/9SmYvia+kCg=,BBzumQNXUHKdEBOB7mAJuzok+IJA1c2Ryg/yzTF6tik=,1
1,Xumu+NIjS6QYVxDS4/t3SawvJ7viT9hPKXmf0RtLNx8=,bhp/MpSNoqoxOIB+/l8WPqu6jldth4DIpCm3ayXnJqM=,1
2,Xumu+NIjS6QYVxDS4/t3SawvJ7viT9hPKXmf0RtLNx8=,JNWfrrC7zNN7BdMpsISKa4Mw+xVJYNnxXh3/Epw7QgY=,1
3,Xumu+NIjS6QYVxDS4/t3SawvJ7viT9hPKXmf0RtLNx8=,2A87tzfnJTSWqD7gIZHisolhe4DMdzkbd6LzO1KHjNs=,1
4,FGtllVqz18RPiwJj/edr2gV78zirAiY/9SmYvia+kCg=,3qm6XTZ6MOCU11x8FIVbAGH5l5uMkT3/ZalWG1oo2Gc=,1


In [18]:
df.shape

(3714656, 3)

In [19]:
df['msno'] = df['msno'].astype('category')

In [20]:
df['song_id'] = df['song_id'].astype('category')

In [21]:
df['user_id'] = df['msno'].cat.codes
df['item_id'] = df['song_id'].cat.codes

In [22]:
user_map = dict(enumerate(df['msno'].cat.categories))
song_map = dict(enumerate(df['song_id'].cat.categories))

In [23]:
df[['msno', 'user_id']].head()
df[['song_id', 'item_id']].head()

,song_id,item_id
0,BBzumQNXUHKdEBOB7mAJuzok+IJA1c2Ryg/yzTF6tik=,46606
1,bhp/MpSNoqoxOIB+/l8WPqu6jldth4DIpCm3ayXnJqM=,139055
2,JNWfrrC7zNN7BdMpsISKa4Mw+xVJYNnxXh3/Epw7QgY=,75249
3,2A87tzfnJTSWqD7gIZHisolhe4DMdzkbd6LzO1KHjNs=,14771
4,3qm6XTZ6MOCU11x8FIVbAGH5l5uMkT3/ZalWG1oo2Gc=,20750


In [24]:
df.groupby("song_id")["item_id"].nunique().max()

C:\Users\BUSE\AppData\Local\Temp\ipykernel_27656\1410588550.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("song_id")["item_id"].nunique().max()


1

In [25]:
n_users = df['user_id'].nunique()
n_items = df['item_id'].nunique()

In [26]:
#sparse interaction matrix csr
interaction_matrix = csr_matrix(
    (df['target'].astype(np.float32), (df['user_id'], df['item_id'])),
    shape=(n_users, n_items)
)

In [27]:
#train test split
train_user_item, test_user_item = train_test_split(interaction_matrix,  train_percentage=0.8) 

In [28]:
print(f"interaction_matrix:", interaction_matrix.shape)
print(f"Train (User-Item) Shape: {train_user_item.shape}")
print(f"Test (User-Item) Shape: {test_user_item.shape}")

interaction_matrix: (27113, 223723)
Train (User-Item) Shape: (27113, 223723)
Test (User-Item) Shape: (27113, 223723)


In [29]:
# initialize a model
model = implicit.als.AlternatingLeastSquares(
    factors=32, 
    regularization=0.1, 
    iterations=15, 
    random_state=42
)

C:\Users\BUSE\anaconda3\Lib\site-packages\implicit\cpu\als.py:95: RuntimeWarning: Intel MKL BLAS is configured to use 14 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'MKL_NUM_THREADS=1' or by callng 'threadpoolctl.threadpool_limits(1, "blas")'. Having MKL use a threadpool can lead to severe performance issues
  check_blas_config()


In [30]:
model.fit(train_user_item)

  0%|          | 0/15 [00:00<?, ?it/s]

In [31]:
print(f"User Factors Matrix: {model.user_factors.shape}")
print(f"Item Factors Matrix: {model.item_factors.shape}")

User Factors Matrix: (27113, 32)
Item Factors Matrix: (223723, 32)


In [33]:
def calculate_precision_at_k(model, train_user_items, test_user_items, k=10):
    # Find users who have interactions in the test set
    test_user_ids = np.where(test_user_items.getnnz(axis=1) > 0)[0]
    
    precision_scores = []

    # Sampling the first 500 users for performance
    for user_id in test_user_ids[:500]:
        # Generate top K recommendations
        recommendations, _ = model.recommend(
            userid=user_id,
            user_items=train_user_items[user_id],
            N=k,
            filter_already_liked_items=True
        )
        
        # Get ground truth from the test set
        actual_items = test_user_items[user_id].indices
        
        # Calculate intersection
        hits = len(set(recommendations) & set(actual_items))
        
        # Add precision for this user
        precision_scores.append(hits / k)

    return np.mean(precision_scores)

# Run the evaluation
final_score = calculate_precision_at_k(model, train_user_item, test_user_item, k=10)
print(f"Precision@10 Score: {final_score:.4f}")

Precision@10 Score: 0.2552
